# Datavidia 11.0 — Task-1 · Notebook Pelatihan Model

Notebook ini adalah **notebook model**: seluruh proses yang menghasilkan berkas model di `models/`
(5 model LightGBM + efek acak Stage-2 + bobot ensemble).

| | |
|---|---|
| Tugas | Klasifikasi 7 kelas `status_penanganan` |
| Metrik | Macro F1 |
| Data latih | `data/raw/train/shipments.csv` (1.243.617 baris) + 7 tabel pendukung |
| Data uji | `data/raw/test/shipments.csv` (362.245 baris) — **tidak dipakai melatih** |

## Arsitektur pipeline

```
data/raw  ──►  build_features.py  ──►  feat_train.pkl / feat_test.pkl   (190 fitur)
                                            │
                                            ▼
                       Stage-1: LightGBM 7-kelas, 5-fold stratified (seed 42)
                       early stopping 100, 2.000 rounds maksimum
                                            │  OOF logits + test logits (rata-rata 5 fold)
                                            ▼
                       Stage-2: GLMM empirical Bayes — efek acak per entitas
                       hub (reg=10) · courier (reg=5) · customer (reg=20)
                       logit_c += u_entitas,c
                                            │
                                            ▼
                       Ensemble: pencarian bobot Stage-1 vs Stage-2 (grid 0,01)
                                            │
                                            ▼
                       submissions/submission_final.csv
```

## Berkas yang dihasilkan notebook ini

| Berkas | Isi |
|---|---|
| `models/lgb_f0..f4.txt` | 5 model LightGBM (satu per fold) |
| `models/feature_spec.json` | daftar 190 fitur, 10 kolom kategorikal + kosakata kategorinya, urutan kelas |
| `models/meta_stage1.json` | kelas, parameter, skor OOF, pengali kelas |
| `models/stage2_{hub,courier,customer}_corr.npy` | tabel koreksi efek acak per entitas |
| `models/ensemble_weight.npy` | bobot Stage-1 pada ensemble (0,0 = murni Stage-2) |
| `models/folds.npy` | penugasan fold tiap baris latih (untuk reproduksi) |

## Waktu eksekusi (2 thread CPU, 32 GB RAM)

| Tahap | Waktu |
|---|---|
| Feature engineering (train+test) | ± 5 menit |
| Stage-1 5-fold LightGBM | ± 8,5 jam |
| Stage-2 GLMM + uji lintas fold + ensemble | ± 12 menit |
| **Total dari nol** | **± 9 jam** |

Angka diukur pada mesin 2 core. Mesin dengan core lebih banyak jauh lebih cepat.

Karena itu sel pelatihan Stage-1 di bawah **tidak dijalankan ulang secara default**
(`RUN_STAGE1 = False`). Berkas model yang disertakan adalah hasil persis dari sel tersebut.
Notebook `02_submission_inference.ipynb` memakai berkas itu dan **tidak melatih apa pun**.

> Catatan kepatuhan: label `status_penanganan` hanya dibaca dari data *train*. Data *test*
> dipakai untuk (a) membangun fitur perilaku entitas tanpa label (transduktif, bukan kebocoran
> label) dan (b) prediksi. Tidak ada label test yang dipakai di tahap mana pun.


## 0. Persiapan lingkungan

In [1]:
import json, os, sys, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

SEED = 42
ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
MODELS = ROOT / "models"

print("ROOT :", ROOT)
print("cwd  :", Path.cwd())

ROOT : C:\Datavidia\deliverable-task1
cwd  : C:\Datavidia\deliverable-task1


In [2]:
import lightgbm as lgb, numpy as np, pandas as pd, sklearn
from scipy.special import softmax
from sklearn.metrics import f1_score, log_loss, classification_report

print("python      ", sys.version.split()[0])
print("lightgbm    ", lgb.__version__)
print("numpy       ", np.__version__)
print("pandas      ", pd.__version__)
print("scikit-learn", sklearn.__version__)
print("\nBerkas model yang disertakan:")
for p in sorted(MODELS.iterdir()):
    print(f"  {p.name:34s} {p.stat().st_size / 1e6:8.1f} MB")

python       3.10.6
lightgbm     4.7.0
numpy        2.2.6
pandas       2.2.3
scikit-learn 1.7.2

Berkas model yang disertakan:
  class_multipliers.npy                   0.0 MB
  ensemble_fit_logits.npy                69.6 MB
  ensemble_weight.npy                     0.0 MB
  feature_spec.json                       0.0 MB
  folds.npy                               1.2 MB
  lgb_f0.txt                             41.9 MB
  lgb_f1.txt                             40.1 MB
  lgb_f2.txt                             43.1 MB
  lgb_f3.txt                             39.5 MB
  lgb_f4.txt                             41.4 MB
  meta_stage1.json                        0.0 MB
  meta_stage2.json                        0.0 MB
  model_summary.json                      0.0 MB
  stage1_oof_logits.npy                  34.8 MB
  stage1_oof_true.npy                     9.9 MB
  stage1_test_logits.npy                 10.1 MB
  stage2_courier_corr.npy                 2.6 MB
  stage2_customer_corr.npy              

## 1. Data

Delapan tabel per split. Hanya `shipments.csv` yang memuat target; tujuh tabel lain adalah
tabel anak (scan_events, route_segments, invoices, promotions, customers, couriers, hubs,
service_packages) yang diagregasi menjadi fitur.

In [3]:
raw = ROOT / "data/raw"
tr_head = pd.read_csv(raw / "train/shipments.csv", nrows=5)
te_head = pd.read_csv(raw / "test/shipments.csv", nrows=5)
n_tr = sum(1 for _ in open(raw / "train/shipments.csv", encoding="utf-8")) - 1
n_te = sum(1 for _ in open(raw / "test/shipments.csv", encoding="utf-8")) - 1

print(f"train shipments : {n_tr:,} baris x {tr_head.shape[1]} kolom")
print(f"test  shipments : {n_te:,} baris x {te_head.shape[1]} kolom")
print("\nkolom shipments:", list(tr_head.columns))

y_raw = pd.read_csv(raw / "train/shipments.csv", usecols=["status_penanganan"])["status_penanganan"]
print("\ndistribusi target (train):")
dist = y_raw.value_counts()
for k, v in dist.items():
    print(f"  {k:24s} {v:8,d}  ({v/len(y_raw):6.2%})")

train shipments : 1,243,617 baris x 38 kolom
test  shipments : 362,245 baris x 37 kolom

kolom shipments: ['shipment_id', 'customer_id', 'courier_id', 'hub_id', 'package_id', 'promo_id', 'weight_kg', 'volume_l', 'declared_value', 'distance_km', 'ship_month', 'ship_dow', 'ship_hour', 'is_cod', 'is_promo', 'channel', 'pickup_type', 'goods_category', 'dest_region', 'insurance_opt', 'qty_items', 'coupon_count', 'gift_wrap', 'fragility_flag', 'order_to_ship_hours', 'prioritas_pengiriman', 'jumlah_catatan', 'flag_alamat_baru', 'berat_volumetrik', 'kota_asal', 'kota_tujuan', 'skor_prioritas_kirim', 'indeks_rute_sekunder', 'faktor_muatan_internal', 'varian_cuaca_rute', 'indeks_waktu_proses', 'skor_kualitas_cek', 'status_penanganan']



distribusi target (train):


  normal                    473,531  (38.08%)
  konfirmasi_alamat         272,911  (21.94%)
  tertunda_operasional      185,496  (14.92%)
  gagal_antar_awal          124,341  (10.00%)
  retur_pengirim             87,740  ( 7.06%)
  rusak_perjalanan           61,942  ( 4.98%)
  klaim_hilang               37,656  ( 3.03%)


## 2. Feature engineering — `src/build_features.py`

190 fitur dari empat kelompok:

1. **Shipment** — berat, volume, nilai deklarasi, jarak, jam kirim, COD, prioritas, jumlah item.
2. **Agregasi tabel anak** — `scan_events` (gap antar scan, kualitas scan, pola manual),
   `route_segments` (delay, durasi), `invoices` (klaim, denda, hari), `promotions` (jumlah kupon).
3. **Profil perilaku entitas tanpa label** — rata-rata perilaku per `hub_id`, `courier_id`,
   `customer_id`, dan `hub × bulan`, dihitung pada gabungan train+test **tanpa memakai label**.
   Untuk courier/customer dipakai *leave-one-out* supaya baris tidak ikut menghitung dirinya.
4. **Deviasi** — selisih perilaku pengiriman terhadap profil courier/hub-nya
   (`shp_vs_co_*`, `co_vs_hu_*`).

Kategorikal (10 kolom) diubah ke `pandas.Categorical` dengan kosakata gabungan train+test,
lalu LightGBM menanganinya sebagai fitur kategorikal asli.

In [4]:
feat_train = ROOT / "data/processed/feat_train.pkl"
feat_test  = ROOT / "data/processed/feat_test.pkl"

if not (feat_train.exists() and feat_test.exists()):
    import build_features
    build_features.main()          # ± 6 menit
else:
    print("Fitur sudah dibangun sebelumnya oleh src/build_features.py — dilewati.")

tr = pd.read_pickle(feat_train)
te = pd.read_pickle(feat_test)
print("train:", tr.shape, "| test:", te.shape)
print("kolom kategorikal:", [c for c in tr.columns if str(tr[c].dtype) == "category"])

Fitur sudah dibangun sebelumnya oleh src/build_features.py — dilewati.


train: (1243617, 196) | test: (362245, 195)
kolom kategorikal: ['cu_segment', 'cu_preferred_payment', 'cu_loyalty_tier', 'co_vehicle_type', 'co_contract_type', 'co_shift_code', 'hu_region', 'hu_hub_type', 'sp_speed_class', 'sp_handling_class']


## 3. Stage-1 — LightGBM 5-fold

- Stratified 5-fold, `random_state=42` → `folds.npy`.
- 7 kelas, `num_class=7`, objective multiclass.
- `learning_rate=0.05`, `num_leaves=127`, `min_data_in_leaf=40`, `feature_fraction=0.7`,
  `bagging_fraction=0.8`, `lambda_l2=2`, `max_bin=63`, `deterministic=True`.
- Early stopping 100 ronde pada multi_logloss, maksimum 2.000 ronde.
- Prediksi test tiap fold dirata-rata (logit), sehingga model akhir = ensemble 5 fold.

Jalankan dengan `RUN_STAGE1 = True` bila ingin melatih ulang dari nol (± 8,5 jam).

In [5]:
RUN_STAGE1 = False          # ubah ke True untuk melatih ulang 5 fold dari nol
EXCLUDE = {"shipment_id", "status_penanganan", "customer_id", "courier_id", "hub_id", "package_id"}


def train_stage1(tr, te):
    classes = sorted(tr["status_penanganan"].unique())
    y = tr["status_penanganan"].map({c: i for i, c in enumerate(classes)}).to_numpy(np.int32)
    K = len(classes)

    feat_cols, cat_cols = [], []
    for c in tr.columns:
        if c in EXCLUDE:
            continue
        if str(tr[c].dtype) == "category":
            cat_cols.append(c)
        elif pd.api.types.is_numeric_dtype(tr[c].dtype):
            feat_cols.append(c)

    X = tr[feat_cols + cat_cols].copy()
    X_test = te[feat_cols + cat_cols].copy()
    for c in feat_cols:
        X[c] = X[c].fillna(0.0)
        X_test[c] = X_test[c].fillna(0.0)

    from sklearn.model_selection import StratifiedKFold
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    folds = np.zeros(len(y), dtype=np.int8)
    for f, (_, va) in enumerate(skf.split(X, y)):
        folds[va] = f
    np.save(ROOT / "models/folds.npy", folds)

    params = {
        "objective": "multiclass", "num_class": K, "metric": "multi_logloss",
        "boosting_type": "gbdt", "learning_rate": 0.05, "num_leaves": 127, "max_depth": -1,
        "min_data_in_leaf": 40, "feature_fraction": 0.7, "bagging_fraction": 0.8,
        "bagging_freq": 1, "lambda_l2": 2.0, "max_bin": 63, "verbosity": -1,
        "num_threads": 2, "seed": SEED, "deterministic": True,
    }

    oof = np.zeros((len(y), K), dtype=np.float32)
    test_logits = np.zeros((len(X_test), K), dtype=np.float32)

    for f in range(5):
        tri, vai = np.where(folds != f)[0], np.where(folds == f)[0]
        dtrain = lgb.Dataset(X.iloc[tri], label=y[tri], categorical_feature=cat_cols)
        dvalid = lgb.Dataset(X.iloc[vai], label=y[vai], reference=dtrain, categorical_feature=cat_cols)
        model = lgb.train(params, dtrain, num_boost_round=2000, valid_sets=[dvalid],
                          callbacks=[lgb.early_stopping(100, verbose=False)])
        oof[vai] = model.predict(X.iloc[vai], num_iteration=model.best_iteration, raw_score=True)
        test_logits += model.predict(X_test, num_iteration=model.best_iteration, raw_score=True) / 5.0
        model.save_model(str(ROOT / f"models/lgb_f{f}.txt"))
        print(f"fold {f}: best_iter={model.best_iteration:4d} "
              f"F1={f1_score(y[vai], oof[vai].argmax(1), average='macro'):.5f}")
        del dtrain, dvalid, model

    np.save(ROOT / "models/stage1_oof_logits.npy", oof)
    np.save(ROOT / "models/stage1_oof_true.npy", y.astype(np.int64))
    np.save(ROOT / "models/stage1_test_logits.npy", test_logits)
    return classes, y, oof, test_logits


if RUN_STAGE1:
    classes, y, oof_logits, test_logits = train_stage1(tr, te)
else:
    classes = json.load(open(MODELS / "meta_stage1.json"))["classes"]
    y = np.load(MODELS / "stage1_oof_true.npy")
    oof_logits = np.load(MODELS / "stage1_oof_logits.npy")
    test_logits = np.load(MODELS / "stage1_test_logits.npy")
    print("Melewati pelatihan — memakai artefak model yang disertakan.")

print(f"{len(classes)} kelas:", classes)
print("OOF logits:", oof_logits.shape, "| test logits:", test_logits.shape)

Melewati pelatihan — memakai artefak model yang disertakan.
7 kelas: ['gagal_antar_awal', 'klaim_hilang', 'konfirmasi_alamat', 'normal', 'retur_pengirim', 'rusak_perjalanan', 'tertunda_operasional']
OOF logits: (1243617, 7) | test logits: (362245, 7)


### 3.1 Skor Stage-1 (out-of-fold, jujur — tidak ada baris yang diprediksi model yang melatihnya)

In [6]:
meta1 = json.load(open(MODELS / "meta_stage1.json"))
print(f"OOF multi_logloss : {meta1['oof_logloss']:.5f}")
print(f"OOF macro F1      : {meta1['oof_argmax_f1']:.5f}")
print(f"F1 dgn pengali    : {meta1['cv_tuned_f1']:.5f}  (pengali kelas dioptimasi pada OOF yang sama → optimistis)")
print("\nper kelas:")
print(classification_report(y, oof_logits.argmax(1), target_names=classes, digits=4, zero_division=0))

OOF multi_logloss : 0.82386
OOF macro F1      : 0.58947
F1 dgn pengali    : 0.60225  (pengali kelas dioptimasi pada OOF yang sama → optimistis)

per kelas:


                      precision    recall  f1-score   support

    gagal_antar_awal     0.5210    0.4119    0.4601    124341
        klaim_hilang     0.5860    0.4021    0.4769     37656
   konfirmasi_alamat     0.6724    0.7172    0.6941    272911
              normal     0.7734    0.8589    0.8139    473531
      retur_pengirim     0.5182    0.3528    0.4198     87740
    rusak_perjalanan     0.5757    0.4440    0.5014     61942
tertunda_operasional     0.7417    0.7796    0.7602    185496

            accuracy                         0.7011   1243617
           macro avg     0.6269    0.5666    0.5895   1243617
        weighted avg     0.6877    0.7011    0.6906   1243617



## 4. Stage-2 — GLMM empirical Bayes (efek acak entitas)

Model: `logit_c(baris) = logit_c^{Stage-1}(baris) + u_{entitas,c}`

dengan `u` adalah rata-rata residual terkecil-kan (`(Σ residual) / (n_entitas + reg)`), residual =
`y_c − p_c^{Stage-1}`. Tiga entitas dipakai berurutan: hub (reg=10), courier (reg=5), customer (reg=20).
Regularisasi makin besar untuk entitas dengan riwayat makin sedikit, sehingga entitas langka tidak
menggeser prediksi. Entitas yang tidak dikenal saat inferensi mendapat koreksi 0.

In [7]:
def fit_glmm_per_class(logits, true, groups, n_classes, reg=1.0):
    """u_entitas,c = sum(y_c - p_c) / (n_entitas + reg)"""
    groups_s = pd.Series(np.asarray(groups)).astype(str)
    probs = softmax(logits, axis=1)
    out = {}
    for c in range(n_classes):
        resid = (true == c).astype(float) - probs[:, c]
        entity_resid = pd.Series(resid).groupby(groups_s.values).sum()
        entity_count = groups_s.value_counts()
        out[c] = (entity_resid / (entity_count + reg)).to_dict()
    return out


def apply_glmm(logits, groups, corrections, n_classes):
    corrected = logits.copy()
    for c in range(n_classes):
        m = corrections[c]
        corrected[:, c] += np.array([m.get(g, 0.0) for g in groups], dtype=np.float32)
    return corrected


REGS = {"hub": 10.0, "courier": 5.0, "customer": 20.0}
G_FIT = {
    "hub": tr["hub_id"].astype(str).values,
    "courier": tr["courier_id"].astype(str).values,
    "customer": tr["customer_id"].astype(str).values,
}
G_TEST = {
    "hub": te["hub_id"].astype(str).values,
    "courier": te["courier_id"].astype(str).values,
    "customer": te["customer_id"].astype(str).values,
}

corr = {n: fit_glmm_per_class(oof_logits, y, G_FIT[n], len(classes), reg=REGS[n]) for n in REGS}

fit_corrected = oof_logits.copy()
test_corrected = test_logits.copy()
for n in ("hub", "courier", "customer"):
    fit_corrected = apply_glmm(fit_corrected, G_FIT[n], corr[n], len(classes))
    test_corrected = apply_glmm(test_corrected, G_TEST[n], corr[n], len(classes))
    print(f"efek {n:9s} diterapkan — entitas: {len(corr[n][0]):,}")

print(f"\nMacro F1 sebelum GLMM : {f1_score(y, oof_logits.argmax(1), average='macro'):.6f}")
print(f"Macro F1 sesudah GLMM : {f1_score(y, fit_corrected.argmax(1), average='macro'):.6f}")

efek hub       diterapkan — entitas: 654


efek courier   diterapkan — entitas: 23,806


efek customer  diterapkan — entitas: 508,086



Macro F1 sebelum GLMM : 0.589473


Macro F1 sesudah GLMM : 0.605382


### 4.1 Uji kejujuran — GLMM di-fit lintas fold

Angka di atas dihitung pada OOF yang **sama** dengan yang dipakai menaksir efek entitas, sehingga
optimistis. Uji yang benar: untuk setiap fold, taksir efek entitas hanya dari 4 fold lain, lalu
terapkan ke fold yang ditahan. Angka inilah estimasi performa yang bisa dipertanggungjawabkan.

In [8]:
folds = np.load(MODELS / "folds.npy")
pred_cf = np.zeros(len(y), dtype=np.int32)
for f in np.unique(folds):
    m_tr, m_va = folds != f, folds == f
    c_f = {n: fit_glmm_per_class(oof_logits[m_tr], y[m_tr], G_FIT[n][m_tr], len(classes), reg=REGS[n])
           for n in REGS}
    block = oof_logits[m_va].copy()
    for n in ("hub", "courier", "customer"):
        block = apply_glmm(block, G_FIT[n][m_va], c_f[n], len(classes))
    pred_cf[m_va] = block.argmax(1)

f1_cf = f1_score(y, pred_cf, average="macro")
f1_base = f1_score(y, oof_logits.argmax(1), average="macro")
print(f"Stage-1 saja (OOF)                 : {f1_base:.6f}")
print(f"Stage-2, efek di-fit pada OOF sama : {f1_score(y, fit_corrected.argmax(1), average='macro'):.6f}  (optimistis)")
print(f"Stage-2, efek lintas fold (jujur)  : {f1_cf:.6f}")
print(f"\nSelisih jujur vs Stage-1: {f1_cf - f1_base:+.6f}")

Stage-1 saja (OOF)                 : 0.589473


Stage-2, efek di-fit pada OOF sama : 0.605382  (optimistis)
Stage-2, efek lintas fold (jujur)  : 0.591851

Selisih jujur vs Stage-1: +0.002378


## 5. Ensemble Stage-1 vs Stage-2

Bobot `w` dicari pada grid 0,01–0,99 di dua ruang (logit dan probabilitas), dipilih yang terbaik
pada OOF. Hasil pencarian menyimpan `w` di `models/ensemble_weight.npy`.

In [9]:
def search_weights(a, b, true, step=0.01):
    best_w, best_f1 = 0.5, -1.0
    for w in np.arange(0.0, 1.0 + step, step):
        f1 = f1_score(true, np.argmax(w * a + (1 - w) * b, axis=1), average="macro")
        if f1 > best_f1:
            best_w, best_f1 = w, f1
    return float(best_w), float(best_f1)

w_logit, f1_logit = search_weights(oof_logits, fit_corrected, y)
w_prob, f1_prob = search_weights(softmax(oof_logits, 1), softmax(fit_corrected, 1), y)
print(f"ruang logit : w={w_logit:.2f}  F1={f1_logit:.6f}")
print(f"ruang prob  : w={w_prob:.2f}  F1={f1_prob:.6f}")

w_saved = float(np.load(MODELS / "ensemble_weight.npy")[0])
print(f"\nbobot tersimpan di models/ensemble_weight.npy: {w_saved:.2f} "
      f"({'murni Stage-2' if w_saved == 0 else 'campuran'})")

p_test = w_saved * softmax(test_logits, 1) + (1 - w_saved) * softmax(test_corrected, 1)
pred_test = p_test.argmax(1)
print("distribusi prediksi test:")
for i, c in enumerate(classes):
    print(f"  {c:24s} {(pred_test == i).mean():6.2%}")

ruang logit : w=0.00  F1=0.605382
ruang prob  : w=0.00  F1=0.605382

bobot tersimpan di models/ensemble_weight.npy: 0.00 (murni Stage-2)


distribusi prediksi test:
  gagal_antar_awal          7.87%
  klaim_hilang              2.07%
  konfirmasi_alamat        23.50%
  normal                   41.68%
  retur_pengirim            4.83%
  rusak_perjalanan          3.93%
  tertunda_operasional     16.11%


## 6. Verifikasi reproduksibilitas

Berkas model dimuat ulang dari disk, lalu diprediksi ulang 20.000 baris latih. Hasilnya harus sama
dengan logits OOF yang tersimpan — bukti bahwa model yang dikumpulkan memang model yang menghasilkan
submission.

In [10]:
idx = np.random.default_rng(0).choice(len(tr), 20_000, replace=False)
Xs = tr.iloc[idx][[c for c in tr.columns if c not in EXCLUDE]].copy()

# urutan kolom mengikuti feature_spec.json (persis urutan saat pelatihan)
spec = json.load(open(MODELS / "feature_spec.json"))
Xs = Xs[spec["feature_cols"]].copy()
for c in spec["cat_cols"]:
    Xs[c] = pd.Categorical(Xs[c].astype(str), categories=spec["categories"][c])
for c in spec["feature_cols"]:
    if c not in spec["cat_cols"]:
        Xs[c] = pd.to_numeric(Xs[c], errors="coerce").astype("float32").fillna(0.0)

rebuilt = np.zeros((len(Xs), len(classes)), dtype=np.float32)
for f in range(5):
    b = lgb.Booster(model_file=str(MODELS / f"lgb_f{f}.txt"))
    rebuilt += b.predict(Xs, raw_score=True) / 5.0

# OOF baris ini berasal dari fold tempat baris tsb ditahan, bukan rata-rata 5 model
fold_of_row = folds[idx]
ref = oof_logits[idx]
print("cek: model fold-nya sendiri vs OOF tersimpan")
ok = 0
for f in range(5):
    m = fold_of_row == f
    if m.sum() == 0:
        continue
    b = lgb.Booster(model_file=str(MODELS / f"lgb_f{f}.txt"))
    pr = b.predict(Xs[m], raw_score=True)
    d = float(np.abs(pr - ref[m]).max())
    print(f"  fold {f}: n={m.sum():5d}  max|Δ|={d:.3e}")
    ok += d < 1e-4
print("\nSEMUA COCOK" if ok == 5 else "\nADA SELISIH — periksa versi library")

cek: model fold-nya sendiri vs OOF tersimpan


  fold 0: n= 4011  max|Δ|=2.384e-07


  fold 1: n= 3925  max|Δ|=2.384e-07


  fold 2: n= 3982  max|Δ|=2.384e-07


  fold 3: n= 4057  max|Δ|=2.384e-07


  fold 4: n= 4025  max|Δ|=2.384e-07

SEMUA COCOK


## 7. Ringkasan hasil

| Tahap | Macro F1 | Sifat |
|---|---|---|
| Stage-1 LightGBM 5-fold | 0,5895 | OOF jujur |
| Stage-2 GLMM (efek di-fit pada OOF yang sama) | 0,6054 | optimistis |
| Stage-2 GLMM (efek lintas fold) | lihat sel 4.1 | jujur |
| Submission final | Stage-2, `w=0` | — |

Yang dikirim ke Kaggle adalah `submission_final.csv` yang dihasilkan
`02_submission_inference.ipynb` dari berkas model di folder `models/`.


In [11]:
summary = {
    "stage1_oof_macro_f1": round(float(meta1["oof_argmax_f1"]), 6),
    "stage2_same_oof_macro_f1": round(float(f1_score(y, fit_corrected.argmax(1), average="macro")), 6),
    "stage2_crossfit_macro_f1": round(float(f1_cf), 6),
    "ensemble_weight_stage1": w_saved,
    "n_features": len(spec["feature_cols"]),
    "n_classes": len(classes),
    "n_train": int(len(y)),
    "n_test": int(len(te)),
}
print(json.dumps(summary, indent=2))
json.dump(summary, open(MODELS / "model_summary.json", "w"), indent=2)
print("\ntersimpan: models/model_summary.json")

{
  "stage1_oof_macro_f1": 0.589473,
  "stage2_same_oof_macro_f1": 0.605382,
  "stage2_crossfit_macro_f1": 0.591851,
  "ensemble_weight_stage1": 0.0,
  "n_features": 190,
  "n_classes": 7,
  "n_train": 1243617,
  "n_test": 362245
}

tersimpan: models/model_summary.json
